In [0]:
# Step 122: Silver pipeline runner with failure logging

import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, LongType
)
from delta.tables import DeltaTable

# Runner parameters
dbutils.widgets.dropdown(
    "load_type",
    "incremental",
    ["full", "incremental_load"]
)
dbutils.widgets.text("processing_date", "")
dbutils.widgets.text("source_folder", "")

load_type = dbutils.widgets.get("load_type")
processing_date = dbutils.widgets.get("processing_date").strip()
source_folder = dbutils.widgets.get("source_folder").strip()

runner_run_id = str(uuid.uuid4())
runner_start_time = datetime.now(timezone.utc).replace(tzinfo=None)

log_table = "workspace.default.pipeline_execution_logs"

try:
    print("Starting Silver pipeline...")

    result = dbutils.notebook.run(
        "./02_silver_pipeline_final",
        0,
        {
            "load_type": load_type,
            "processing_date": processing_date,
            "source_folder": source_folder
        }
    )

    print("Silver pipeline completed successfully!")
    print("Notebook result:", result)

except Exception as e:
    runner_end_time = datetime.now(timezone.utc).replace(tzinfo=None)

    log_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("layer", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("load_type", StringType(), True),
        StructField("processing_date", StringType(), True),
        StructField("source_folder", StringType(), True),
        StructField("start_timestamp", TimestampType(), True),
        StructField("end_timestamp", TimestampType(), True),
        StructField("status", StringType(), True),
        StructField("rows_read", LongType(), True),
        StructField("rows_inserted", LongType(), True),
        StructField("rows_updated", LongType(), True),
        StructField("rows_quarantined", LongType(), True),
        StructField("error_message", StringType(), True)
    ])

    failure_data = [(
        runner_run_id,
        "silver",
        "workspace.default.silver_market_observation",
        load_type,
        processing_date or None,
        source_folder or None,
        runner_start_time,
        runner_end_time,
        "FAILED",
        0,
        0,
        0,
        0,
        str(e)[:4000]
    )]

    failure_df = spark.createDataFrame(
        failure_data,
        schema=log_schema
    )

    (
        DeltaTable.forName(spark, log_table)
        .alias("target")
        .merge(
            failure_df.alias("source"),
            "target.run_id = source.run_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Silver pipeline FAILED. Failure recorded in execution logs.")
    raise

In [0]:
# Step 123: Verify the latest Silver execution log

from pyspark.sql import functions as F

silver_logs = (
    spark.table("workspace.default.pipeline_execution_logs")
    .filter(F.col("layer") == "silver")
    .orderBy(F.col("end_timestamp").desc())
)

display(
    silver_logs.select(
        "run_id",
        "load_type",
        "status",
        "rows_read",
        "rows_inserted",
        "rows_updated",
        "rows_quarantined",
        "start_timestamp",
        "end_timestamp"
    ).limit(5)
)

In [0]:
# Step 124: Final Bronze and Silver verification

from pyspark.sql import functions as F

bronze_final = spark.table(
    "workspace.default.bronze_coin_market_raw"
)

silver_final = spark.table(
    "workspace.default.silver_market_observation"
)

quarantine_final = spark.table(
    "workspace.default.silver_quarantine"
)

# Count records
bronze_count = bronze_final.count()
silver_count = silver_final.count()
quarantine_count = quarantine_final.count()

# Check duplicate Silver business keys
duplicate_count = (
    silver_final
    .groupBy("coin_id", "observation_timestamp")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Bronze records:", bronze_count)
print("Silver records:", silver_count)
print("Silver quarantine records:", quarantine_count)
print("Duplicate Silver business keys:", duplicate_count)

if (
    bronze_count == 551996
    and silver_count == 551991
    and quarantine_count == 5
    and duplicate_count == 0
):
    print("PASS: Final Bronze and Silver verification successful!")
else:
    print("CHECK REQUIRED: One or more results differ from expected values.")